In [2]:
import os
os.environ["LD_LIBRARY_PATH"] = (
    "/home/abhim21/emg2qwerty/.venv/lib/python3.10/site-packages/nvidia/cudnn/lib:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)

In [7]:
# Single-user training
!python -m emg2qwerty.train \
  user="single_user" \
  trainer.accelerator=gpu trainer.devices=1 \
  +trainer.precision=16 \
  batch_size=128 \
  num_workers=7 \
  checkpoint=logs/2026-03-08/09-27-18/checkpoints/last.ckpt
  # --multirun

[2026-03-08 09:30:49,519][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

In [10]:
# Single-user testing
!python -m emg2qwerty.train \
  user="single_user" \
  checkpoint="'/home/abhim21/emg2qwerty/logs/2026-03-08/09-30-49/checkpoints/epoch=122-step=3690.ckpt'" \
  train=False \
  trainer.accelerator=gpu \
  decoder=ctc_greedy \
  hydra.launcher.mem_gb=64 \
  # --multirun

[2026-03-08 11:45:29,667][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

## Experiment 1: Number of Electrode Channels vs CER

Vary the number of electrode channels (1, 2, 4, 8, 16) and measure test CER.

Baseline (16 channels): `test/CER ≈ 22.1%` from the single-user TDS Conv model.

Each run uses a `ChannelSubset` transform that keeps only the first N channels per band, and `num_electrode_channels=N` to resize `SpectrogramNorm`. `in_features = N * 33`.

In [ ]:
# 8 channels  (in_features = 8 * 33 = 264)
!python -m emg2qwerty.train \
  user=single_user transforms=channels_8 \
  module.num_electrode_channels=8 module.in_features=264 \
  trainer.accelerator=gpu trainer.devices=1 \
  +trainer.precision=16 batch_size=128 num_workers=7


In [ ]:
# 4 channels  (in_features = 4 * 33 = 132)
!python -m emg2qwerty.train \
  user=single_user transforms=channels_4 \
  module.num_electrode_channels=4 module.in_features=132 \
  trainer.accelerator=gpu trainer.devices=1 \
  +trainer.precision=16 batch_size=128 num_workers=7


In [ ]:
# 2 channels  (in_features = 2 * 33 = 66)
!python -m emg2qwerty.train \
  user=single_user transforms=channels_2 \
  module.num_electrode_channels=2 module.in_features=66 \
  trainer.accelerator=gpu trainer.devices=1 \
  +trainer.precision=16 batch_size=128 num_workers=7


In [ ]:
# 1 channel  (in_features = 1 * 33 = 33)
!python -m emg2qwerty.train \
  user=single_user transforms=channels_1 \
  module.num_electrode_channels=1 module.in_features=33 \
  trainer.accelerator=gpu trainer.devices=1 \
  +trainer.precision=16 batch_size=128 num_workers=7


In [ ]:
import matplotlib.pyplot as plt

# Fill in test/CER values from the run outputs above (and the baseline 16-ch run)
channels = [1,    2,    4,    8,    16]
test_cer = [None, None, None, None, 22.09]  # replace None with actual results

plt.figure(figsize=(6, 4))
plt.plot(channels, test_cer, marker='o')
plt.xlabel('Electrode channels per band')
plt.ylabel('Test CER (%)')
plt.title('CER vs Number of Electrode Channels')
plt.xticks(channels)
plt.grid(True)
plt.tight_layout()
plt.savefig('channels_vs_cer.png', dpi=150)
plt.show()


## Experiment 2: Training Data Amount vs CER

Vary the number of training sessions (1, 2, 4, 8, 16) and measure test CER.

The same model architecture and transforms (16 channels, 2 kHz) are used. Only the `user` config changes to restrict training data.

In [ ]:
# 8 sessions
!python -m emg2qwerty.train \
  user=single_user_8sess \
  trainer.accelerator=gpu trainer.devices=1 \
  +trainer.precision=16 batch_size=128 num_workers=7


In [ ]:
# 4 sessions
!python -m emg2qwerty.train \
  user=single_user_4sess \
  trainer.accelerator=gpu trainer.devices=1 \
  +trainer.precision=16 batch_size=128 num_workers=7


In [ ]:
# 2 sessions
!python -m emg2qwerty.train \
  user=single_user_2sess \
  trainer.accelerator=gpu trainer.devices=1 \
  +trainer.precision=16 batch_size=128 num_workers=7


In [ ]:
# 1 session
!python -m emg2qwerty.train \
  user=single_user_1sess \
  trainer.accelerator=gpu trainer.devices=1 \
  +trainer.precision=16 batch_size=128 num_workers=7


In [ ]:
import matplotlib.pyplot as plt

# Fill in test/CER values from the run outputs above
num_sessions = [1,    2,    4,    8,    16]
test_cer     = [None, None, None, None, 22.09]  # replace None with actual results

plt.figure(figsize=(6, 4))
plt.plot(num_sessions, test_cer, marker='o')
plt.xlabel('Number of training sessions')
plt.ylabel('Test CER (%)')
plt.title('CER vs Amount of Training Data')
plt.xticks(num_sessions)
plt.grid(True)
plt.tight_layout()
plt.savefig('data_amount_vs_cer.png', dpi=150)
plt.show()


## Experiment 3: Sampling Rate vs CER

Vary the effective EMG sampling rate (250, 500, 1000, 2000 Hz) by decimating the raw signal before computing the log-spectrogram.  2000 Hz is the baseline.

The `Downsample(factor=F)` transform is inserted after `ToTensor` and before `LogSpectrogram`.  STFT parameters (n_fft=64, hop_length=16) are unchanged in *samples*, so a lower sampling rate means a wider time window per frame and a lower Nyquist frequency, producing a shorter feature sequence for the same recording.

In [ ]:
# 1000 Hz  (downsample by 2)
!python -m emg2qwerty.train \
  user=single_user transforms=sampling_1000hz \
  trainer.accelerator=gpu trainer.devices=1 \
  +trainer.precision=16 batch_size=128 num_workers=7


In [ ]:
# 500 Hz  (downsample by 4)
!python -m emg2qwerty.train \
  user=single_user transforms=sampling_500hz \
  trainer.accelerator=gpu trainer.devices=1 \
  +trainer.precision=16 batch_size=128 num_workers=7


In [ ]:
# 250 Hz  (downsample by 8)
!python -m emg2qwerty.train \
  user=single_user transforms=sampling_250hz \
  trainer.accelerator=gpu trainer.devices=1 \
  +trainer.precision=16 batch_size=128 num_workers=7


In [ ]:
import matplotlib.pyplot as plt

# Fill in test/CER values from the run outputs above
sampling_hz = [250,  500,  1000, 2000]
test_cer    = [None, None, None, 22.09]  # replace None with actual results

plt.figure(figsize=(6, 4))
plt.plot(sampling_hz, test_cer, marker='o')
plt.xlabel('Effective sampling rate (Hz)')
plt.ylabel('Test CER (%)')
plt.title('CER vs EMG Sampling Rate')
plt.xticks(sampling_hz)
plt.grid(True)
plt.tight_layout()
plt.savefig('sampling_rate_vs_cer.png', dpi=150)
plt.show()
